## Görev 1

In [17]:
# Metni byte'lara çevir. Türkçe harfler içeren bir cümle seç (ç, ğ, ı, ö, ş, ü). Her harfin Unicode kod noktasını ve UTF-8 byte'larını yazdır. Hangi harfler 1 byte, hangileri 2 byte tutuyor? Kod noktalarını doğrudan vocab olarak kullanmak neden kötü bir fikir, kendi cümlelerinle yaz. (0:14:56 - 0:23:50)

Tokenization is at the heart of much weirdness of LLMs. Do not brush it off.

- Why can't LLM spell words? **Tokenization**.
- Why can't LLM do super simple string processing tasks like reversing a string? **Tokenization**.
- Why is LLM worse at non-English languages (e.g. Japanese)? **Tokenization**.
- Why is LLM bad at simple arithmetic? **Tokenization**.
- Why did GPT-2 have more than necessary trouble coding in Python? **Tokenization**.
- Why did my LLM abruptly halt when it sees the string "<|endoftext|>"? **Tokenization**.
- What is this weird warning I get about a "trailing whitespace"? **Tokenization**.
- Why the LLM break if I ask it about "SolidGoldMagikarp"? **Tokenization**.
- Why should I prefer to use YAML over JSON with LLMs? **Tokenization**.
- Why is LLM not actually end-to-end language modeling? **Tokenization**.
- What is the real root of suffering? **Tokenization**.

In [18]:
[ord(x)for x in "Çiçekçi ışığı söndürüp şöyle düşündü: ağaç ne kadar güzelmiş."]

[199,
 105,
 231,
 101,
 107,
 231,
 105,
 32,
 305,
 351,
 305,
 287,
 305,
 32,
 115,
 246,
 110,
 100,
 252,
 114,
 252,
 112,
 32,
 351,
 246,
 121,
 108,
 101,
 32,
 100,
 252,
 351,
 252,
 110,
 100,
 252,
 58,
 32,
 97,
 287,
 97,
 231,
 32,
 110,
 101,
 32,
 107,
 97,
 100,
 97,
 114,
 32,
 103,
 252,
 122,
 101,
 108,
 109,
 105,
 351,
 46]

In [19]:
list("Çiçekçi ışığı söndürüp şöyle düşündü: ağaç ne kadar güzelmiş.".encode("utf-8"))

[195,
 135,
 105,
 195,
 167,
 101,
 107,
 195,
 167,
 105,
 32,
 196,
 177,
 197,
 159,
 196,
 177,
 196,
 159,
 196,
 177,
 32,
 115,
 195,
 182,
 110,
 100,
 195,
 188,
 114,
 195,
 188,
 112,
 32,
 197,
 159,
 195,
 182,
 121,
 108,
 101,
 32,
 100,
 195,
 188,
 197,
 159,
 195,
 188,
 110,
 100,
 195,
 188,
 58,
 32,
 97,
 196,
 159,
 97,
 195,
 167,
 32,
 110,
 101,
 32,
 107,
 97,
 100,
 97,
 114,
 32,
 103,
 195,
 188,
 122,
 101,
 108,
 109,
 105,
 197,
 159,
 46]

In [20]:
"""
195, 135  → Ç       (2 byte)
105       → i       (1 byte)         türkçe karakterlerde 2 byte ingilizce karakterler 1 byte 
195, 167  → ç       (2 byte)
101       → e       (1 byte)
107       → k       (1 byte)
195, 167  → ç       (2 byte)
105       → i       (1 byte)
32        → boşluk  (1 byte)
"""

'\n195, 135  → Ç       (2 byte)\n105       → i       (1 byte)         türkçe karakterlerde 2 byte ingilizce karakterler 1 byte \n195, 167  → ç       (2 byte)\n101       → e       (1 byte)\n107       → k       (1 byte)\n195, 167  → ç       (2 byte)\n105       → i       (1 byte)\n32        → boşluk  (1 byte)\n'

In [21]:
# Kod noktası 128'den küçük olan karakterler boşluk, nokta ve ingilizde karakterde UTF-8'de 1 byte tutuyor.
# Türkçeye özgü ç, ğ, ı, ö, ş, ü karakterlerde kod noktaları 128'den büyük olduğu için 2 byte tutuyor.
# Örneğin Ç'nin kod noktası 199, UTF-8'de 195 ve 135 olmak üzere iki byte'a bölünüyor.
# "Çiçekçi" kelimesi 7 harf ama 10 byte, çünkü içindeki üç tane Ç/ç harfinin her biri 2 byte tutuyor.
# Bu yüzden Türkçe bir metnin byte sayısı karakter sayısından daha fazla oluyor.
#
# Kod noktalarını doğrudan vocab olarak kullanmak kötü bir fikir, çünkü Unicode'da yaklaşık 150 bin
# karakter var. Her birini ayrı bir token yaparsak vocab çok büyük olacak karpathy videoda da bahsediyor ve modelin embedding tablosu da
# büyür. Bu karakterlerin çoğu metinde neredeyse hiç geçmediği için model onları iyi öğrenemez. Bir de 
# unicode sabit olmadığı için yani sürekli değişken olduğu için güvenemeyiz.

## Görev 2

In [22]:
# BPE'yi sıfırdan yaz ve kendi Türkçe metninde eğit. Geçen hafta derlediğin Türkçe metni kullan. Ardışık ikilileri say, en sık geçeni yeni bir token'la birleştir, istediğin vocab boyutuna gelene kadar tekrarla. İlk 20 birleştirmeyi metin olarak yazdır: hangi harf grupları önce birleşiyor? Sıkıştırma oranını (byte sayısı / token sayısı) yaz. (0:23:50 - 0:39:20)

In [23]:
with open('turkce_corpus.txt', 'r', encoding='utf-8') as f:
    text_tr = f.read()

In [24]:
print("metin uzunluğu:", len(text_tr))

metin uzunluğu: 711253


In [25]:
tokens = text_tr.encode("utf-8") # raw bytes
tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 for convenience
print('---')
print(text_tr[:500])
print("length:", len(text_tr))
print('---')
print(tokens[:100])
print("length:", len(tokens))

---
Minimini, güzel, Bedia'yı ailesi büyük bir adama vermek istiyordu. Halbuki o iki senedir, tıbbiye talebesinden olan kuzeni Namık'la işi pişirmişti. Kendini almayı arzu eden bu büyük adam tek gözlüklü, şık bir büyükelçiydi. "Kırkında var, yok..." diyorlardı. Bedia daha on yedisine girmemişti. Annesinin, babasının, hanımninesinin ısrarlarına biraz karşı geldi. Ağladı, sızladı, amma sonunda mağlup oldu. 
:— Ben koca herifi ne yapayım? Elli sene Avrupa'da balolarda sürtmüş! dedikçe,
:— Haltetmişsin!
length: 711253
---
[77, 105, 110, 105, 109, 105, 110, 105, 44, 32, 103, 195, 188, 122, 101, 108, 44, 32, 66, 101, 100, 105, 97, 39, 121, 196, 177, 32, 97, 105, 108, 101, 115, 105, 32, 98, 195, 188, 121, 195, 188, 107, 32, 98, 105, 114, 32, 97, 100, 97, 109, 97, 32, 118, 101, 114, 109, 101, 107, 32, 105, 115, 116, 105, 121, 111, 114, 100, 117, 46, 32, 72, 97, 108, 98, 117, 107, 105, 32, 111, 32, 105, 107, 105, 32, 115, 101, 110, 101, 100, 105, 114, 44, 32, 116, 196, 177, 98, 98, 105]
length:

In [26]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]): # Pythonic way to iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(tokens)
#print(stats)
print(sorted(((v,k) for k,v in stats.items()), reverse=True))

[(28385, (196, 177)), (11676, (110, 32)), (11043, (195, 188)), (10380, (97, 114)), (9877, (32, 98)), (9280, (46, 32)), (9082, (197, 159)), (9014, (101, 32)), (8981, (44, 32)), (8412, (108, 97)), (8326, (101, 114)), (7842, (105, 110)), (7754, (97, 32)), (7727, (97, 110)), (7513, (105, 32)), (7442, (100, 97)), (7122, (32, 107)), (6963, (108, 101)), (6954, (114, 32)), (6757, (105, 114)), (6567, (101, 110)), (6313, (177, 110)), (5889, (100, 101)), (5822, (195, 167)), (5753, (107, 97)), (5685, (98, 105)), (5651, (32, 100)), (5604, (100, 105)), (5142, (32, 103)), (4962, (196, 159)), (4865, (97, 108)), (4839, (97, 107)), (4810, (32, 115)), (4719, (121, 97)), (4619, (177, 32)), (4581, (32, 97)), (4507, (114, 97)), (4479, (114, 100)), (4462, (110, 100)), (4220, (114, 105)), (4206, (109, 97)), (4153, (97, 115)), (4060, (111, 114)), (3881, (32, 121)), (3878, (107, 32)), (3718, (32, 195)), (3653, (121, 111)), (3555, (109, 101)), (3547, (32, 105)), (3539, (46, 46)), (3523, (32, 104)), (3522, (100, 

In [27]:
chr(196), chr(177)

('Ä', '±')

In [28]:
top_pair = max(stats, key=stats.get)
top_pair

(196, 177)

In [29]:
def merge(ids, pair, idx):
  # in the list of ints (ids), replace all consecutive occurences of pair with the new token idx
  newids = []
  i = 0
  while i < len(ids):
    # if we are not at the very last position AND the pair matches, replace it
    if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
      newids.append(idx)
      i += 2
    else:
      newids.append(ids[i])
      i += 1
  return newids

print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

tokens2 = merge(tokens, top_pair, 256)
print(tokens2[:100])
print("length:", len(tokens2))

[5, 6, 99, 9, 1]
[77, 105, 110, 105, 109, 105, 110, 105, 44, 32, 103, 195, 188, 122, 101, 108, 44, 32, 66, 101, 100, 105, 97, 39, 121, 256, 32, 97, 105, 108, 101, 115, 105, 32, 98, 195, 188, 121, 195, 188, 107, 32, 98, 105, 114, 32, 97, 100, 97, 109, 97, 32, 118, 101, 114, 109, 101, 107, 32, 105, 115, 116, 105, 121, 111, 114, 100, 117, 46, 32, 72, 97, 108, 98, 117, 107, 105, 32, 111, 32, 105, 107, 105, 32, 115, 101, 110, 101, 100, 105, 114, 44, 32, 116, 256, 98, 98, 105, 121, 101]
length: 751519


In [30]:
vocab_size = 276 # the desired final vocabulary size
num_merges = vocab_size - 256
ids = list(tokens) # copy so we don't destroy the original list

merges = {} # (int, int) -> int
for i in range(num_merges):
  stats = get_stats(ids)
  pair = max(stats, key=stats.get)
  idx = 256 + i
  print(f"merging {pair} into a new token {idx}")
  ids = merge(ids, pair, idx)
  merges[pair] = idx

merging (196, 177) into a new token 256
merging (110, 32) into a new token 257
merging (195, 188) into a new token 258
merging (97, 114) into a new token 259
merging (46, 32) into a new token 260
merging (197, 159) into a new token 261
merging (101, 32) into a new token 262
merging (44, 32) into a new token 263
merging (101, 114) into a new token 264
merging (97, 32) into a new token 265
merging (105, 32) into a new token 266
merging (105, 114) into a new token 267
merging (195, 167) into a new token 268
merging (105, 110) into a new token 269
merging (100, 97) into a new token 270
merging (196, 159) into a new token 271
merging (97, 108) into a new token 272
merging (97, 110) into a new token 273
merging (256, 32) into a new token 274
merging (97, 107) into a new token 275


In [31]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length: 779904
ids length: 611770
compression ratio: 1.27X


In [32]:
vocab = {i: bytes([i]) for i in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

for (p0, p1), idx in merges.items():
    sol0 = chr(p0) if p0 < 256 else vocab[p0].decode("utf-8")
    sol1 = chr(p1)
    print(sol0, sol1, "into new token ->", vocab[idx].decode("utf-8"))

Ä ± into new token -> ı
n   into new token -> n 
Ã ¼ into new token -> ü
a r into new token -> ar
.   into new token -> . 
Å  into new token -> ş
e   into new token -> e 
,   into new token -> , 
e r into new token -> er
a   into new token -> a 
i   into new token -> i 
i r into new token -> ir
Ã § into new token -> ç
i n into new token -> in
d a into new token -> da
Ä  into new token -> ğ
a l into new token -> al
a n into new token -> an
ı   into new token -> ı 
a k into new token -> ak


In [33]:
# Türkçe metinde en çok geçen karakterler.
from collections import Counter
for harf, adet in Counter(text_tr).most_common(10):
    print(repr(harf), adet)

' ' 90185
'a' 67152
'e' 49904
'i' 47696
'r' 39646
'n' 38250
'l' 30281
'd' 28439
'ı' 28385
'k' 24707


In [34]:
# 20 birleştimede önce Türkçe harf olarak en çok geçen ı nın bytelarının birleştirilmesini görüyoruz çünkü türkçe harfde iki byte olduğu için ilk onu merge ettik.
# Sonra n ve boşluğun birleşiminden muhtemelen n ile biten ifadelerin çok olmasından kaynalı birleştime var.  
# Bir de sık geçen harf grupları var: ar, er, ir, in, da, al, an, ak.
# 19. birleştirme ı önceki birleştirmenin üstüne merge ediliyor: 256 iken boşluk ile birleşip 274 oluyor.
# Sıkıştırma oranı: 779904 byte / 611770 token = 1.27X
# 779904 - 611770 = 168134 yani bpe ile iki tokenı birleştidikten sonra her token başı byte artık 1.27. bu değer en başta 1 di ve dizi uzunluğu da azalmış oldu.

## Görev 3

In [35]:
# encode ve decode yaz. decode(encode(metin)) metnin kendisini vermeli, Türkçe karakterli ve emoji içeren bir örnekle göster. Bazı token dizileri geçerli UTF-8 değil: decode bu durumda ne yapmalı, neden? (0:42:47 - 0:57:36)

In [36]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
  # given ids (list of integers), return Python string
  tokens = b"".join(vocab[idx] for idx in ids)
  text = tokens.decode("utf-8", errors="replace")
  return text

print(decode([128]))

�


In [37]:
merges

{(196, 177): 256,
 (110, 32): 257,
 (195, 188): 258,
 (97, 114): 259,
 (46, 32): 260,
 (197, 159): 261,
 (101, 32): 262,
 (44, 32): 263,
 (101, 114): 264,
 (97, 32): 265,
 (105, 32): 266,
 (105, 114): 267,
 (195, 167): 268,
 (105, 110): 269,
 (100, 97): 270,
 (196, 159): 271,
 (97, 108): 272,
 (97, 110): 273,
 (256, 32): 274,
 (97, 107): 275}

In [38]:
def encode(text):
  # given a string, return list of integers (the tokens)
  tokens = list(text.encode("utf-8"))
  while len(tokens) >= 2:
    stats = get_stats(tokens)
    pair = min(stats, key=lambda p: merges.get(p, float("inf")))
    if pair not in merges:
      break # nothing else can be merged
    idx = merges[pair]
    tokens = merge(tokens, pair, idx)
  return tokens

print(encode("Bugün hava çok güzel, parkta yürüyüşe gidelim! 😊🌞"))

[66, 117, 103, 258, 257, 104, 97, 118, 265, 268, 111, 107, 32, 103, 258, 122, 101, 108, 263, 112, 259, 107, 116, 265, 121, 258, 114, 258, 121, 258, 261, 262, 103, 105, 100, 101, 108, 105, 109, 33, 32, 240, 159, 152, 138, 240, 159, 140, 158]


In [39]:
print(decode(encode("Bugün hava çok güzel, parkta yürüyüşe gidelim! 😊🌞")))

Bugün hava çok güzel, parkta yürüyüşe gidelim! 😊🌞


In [40]:
text2 = decode(encode("Bugün hava çok güzel, parkta yürüyüşe gidelim! 😊🌞"))
print(text2 == "Bugün hava çok güzel, parkta yürüyüşe gidelim! 😊🌞")

True


In [41]:
# Bazı token dizileri geçerli UTF-8 değil çünkü örneğin türkçe karakterdeki gibi 2 bytelık ya da emojideki gibi 4 bytelık veirler anlamsız kalabiliyor birleşince. 
# Decode bunun için error = 'replace' kullanmalı çünkü modelin 100 token ürettiğini düşünelim sırf 1 harf çözümleyemedi diye 99 tokenı kaybetmek istemeyiz. 
# Token'ın yerine � karakterini koyuyor.

## Görev 4

In [42]:
# GPT-4'ün regex desenini ekle. Metni önce GPT-4'ün deseniyle parçalara böl, BPE'yi her parçada ayrı çalıştır. Tokenizer'ı yeniden eğit, birleştirmeleri öncekiyle karşılaştır. Sonra "Ankara'dan" ve "İstanbul'da" gibi kesme işaretli kelimeleri bu desenle böl. Desen İngilizce için yazılmış: Türkçede ne yapıyor, kendi cümlelerinle yaz. (0:57:36 - 1:11:38)

In [43]:
import regex as re
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2pat, "Bugün hava çok güzel, parkta yürüyüşe gidelim! 😊🌞"))

['Bugün', ' hava', ' çok', ' güzel', ',', ' parkta', ' yürüyüşe', ' gidelim', '!', ' 😊🌞']


In [44]:
gpt4pat = re.compile(r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+""")

print(re.findall(gpt4pat, "Bugün hava çok güzel, parkta yürüyüşe gidelim! 😊🌞"))

['Bugün', ' hava', ' çok', ' güzel', ',', ' parkta', ' yürüyüşe', ' gidelim', '!', ' 😊🌞']


In [45]:
example = """
for i in range(1, 101):
    if i % 3 == 0 and i % 5 == 0:
        print("FizzBuzz")
    elif i % 3 == 0:
        print("Fizz")
    elif i % 5 == 0:
        print("Buzz")
    else:
        print(i)
"""
print(re.findall(gpt2pat, example))

['\n', 'for', ' i', ' in', ' range', '(', '1', ',', ' 101', '):', '\n   ', ' if', ' i', ' %', ' 3', ' ==', ' 0', ' and', ' i', ' %', ' 5', ' ==', ' 0', ':', '\n       ', ' print', '("', 'FizzBuzz', '")', '\n   ', ' elif', ' i', ' %', ' 3', ' ==', ' 0', ':', '\n       ', ' print', '("', 'Fizz', '")', '\n   ', ' elif', ' i', ' %', ' 5', ' ==', ' 0', ':', '\n       ', ' print', '("', 'Buzz', '")', '\n   ', ' else', ':', '\n       ', ' print', '(', 'i', ')', '\n']


In [46]:
import tiktoken

# GPT-2 (does not merge spaces)
enc = tiktoken.get_encoding("gpt2")
print(enc.encode("    hello world!!!"))

# GPT-4 (merges spaces)
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode("    hello world!!!"))

[220, 220, 220, 23748, 995, 10185]
[262, 24748, 1917, 12340]


In [47]:
# Metni önce GPT-4'ün deseniyle parçalar bölüyoruz.
chunks = re.findall(gpt4pat, text_tr)
print("parça sayısı:", len(chunks))
print(chunks[:15])
print("parçaları birleştirince metin aynı mı:", "".join(chunks) == text_tr)

parça sayısı: 126769
['Minimini', ',', ' güzel', ',', ' Bedia', "'yı", ' ailesi', ' büyük', ' bir', ' adama', ' vermek', ' istiyordu', '.', ' Halbuki', ' o']
parçaları birleştirince metin aynı mı: True


In [48]:
ids_list = [list(chunk.encode("utf-8")) for chunk in chunks]
n_bytes = sum(len(c) for c in ids_list)

merges2 = {} 
for i in range(num_merges): 
    stats = {}
    for chunk_ids in ids_list:
        for pair in zip(chunk_ids, chunk_ids[1:]):
            stats[pair] = stats.get(pair, 0) + 1
    pair = max(stats, key=stats.get)
    idx = 256 + i
    ids_list = [merge(chunk_ids, pair, idx) for chunk_ids in ids_list]
    merges2[pair] = idx

n_tokens2 = sum(len(c) for c in ids_list)
print("byte sayısı:", n_bytes)
print("token sayısı:", n_tokens2)
print(f"compression ratio: {n_bytes / n_tokens2:.2f}X")

byte sayısı: 779904
token sayısı: 620184
compression ratio: 1.26X


In [49]:
vocab_a = {i: bytes([i]) for i in range(256)}
for (p0, p1), idx in merges.items():
    vocab_a[idx] = vocab_a[p0] + vocab_a[p1]

vocab_b = {i: bytes([i]) for i in range(256)}
for (p0, p1), idx in merges2.items():
    vocab_b[idx] = vocab_b[p0] + vocab_b[p1]

print("normal bpe  GPT-4 desenli")
for i, (idx_a, idx_b) in enumerate(zip(merges.values(), merges2.values()), 1):
    print(f"{i:2d} {vocab_a[idx_a].decode('utf-8')!r:10} {vocab_b[idx_b].decode('utf-8')!r}")

ortak = {vocab_a[i].decode('utf-8') for i in merges.values()} & {vocab_b[i].decode('utf-8') for i in merges2.values()}
print("iki listede de olanlar:", ortak)
print(f"desensiz: {len(tokens) / len(ids):.2f}X, desenli: {n_bytes / n_tokens2:.2f}X")

normal bpe  GPT-4 desenli
 1 'ı'        'ı'
 2 'n '       'ü'
 3 'ü'        'ar'
 4 'ar'       ' b'
 5 '. '       'ş'
 6 'ş'        'er'
 7 'e '       'in'
 8 ', '       'an'
 9 'er'       ' k'
10 'a '       'ir'
11 'i '       'en'
12 'ir'       'ın'
13 'ç'        'ç'
14 'in'       ' d'
15 'da'       ' g'
16 'ğ'        'ğ'
17 'al'       'al'
18 'an'       'ak'
19 'ı '       ' s'
20 'ak'       'di'
iki listede de olanlar: {'ş', 'ı', 'in', 'ak', 'ir', 'ar', 'ü', 'er', 'an', 'ç', 'ğ', 'al'}
desensiz: 1.27X, desenli: 1.26X


In [50]:
# Kesme işaretli kelimeleri GPT-2 deseniyle bölüyoruz.
for kelime in ["Ankara'dan", "İstanbul'da", "Bedia'yı", "Namık'la", "Ali'nin", "Ankara’dan"]:
    print(kelime, "->", re.findall(gpt2pat, kelime))

Ankara'dan -> ['Ankara', "'d", 'an']
İstanbul'da -> ['İstanbul', "'d", 'a']
Bedia'yı -> ['Bedia', "'", 'yı']
Namık'la -> ['Namık', "'", 'la']
Ali'nin -> ['Ali', "'", 'nin']
Ankara’dan -> ['Ankara', '’', 'dan']


In [51]:
# Kesme işaretli kelimeleri GPT-4 deseniyle bölüyoruz.
for kelime in ["Ankara'dan", "İstanbul'da", "Bedia'yı", "Namık'la", "Ali'nin", "Ankara’dan"]:
    print(kelime, "->", re.findall(gpt4pat, kelime))

Ankara'dan -> ['Ankara', "'d", 'an']
İstanbul'da -> ['İstanbul', "'d", 'a']
Bedia'yı -> ['Bedia', "'yı"]
Namık'la -> ['Namık', "'la"]
Ali'nin -> ['Ali', "'nin"]
Ankara’dan -> ['Ankara', '’dan']


In [52]:
# Desenli olanda boşluk, kendinden sonra gelen kelimeyle aynı parçaya giriyor. Örneğin "ın bir" iki parçaya bölünüyor: 'ın' ve ' bir'.
# Bu yüzden "ın" kelimesinin sonundaki n harfi ile boşluk farklı parçalarda kalıyor ve hiç yan yana gelmiyor.
# "n ", "e ", "a ", "i ", "ı " gibi harf + boşluk birleşmeleri yok oldu, çünkü harf ile boşluk artık ayrı parçalarda.
# Türkçede -dan ve -da eki de d ile başladığı için "Ankara'dan" -> ['Ankara', "'d", 'an'] ve "İstanbul'da" -> ['İstanbul', "'d", 'a'] oluyor.
# Yani 'd İngilizce kısaltma sanılıp ayrılıyor, an ve a geride kalıyor. Ek, yanlış yerden kesiliyor, 'dan normalde bütün kalmalıydı.

## Görev 5

In [53]:
# Türkçe ile İngilizceyi karşılaştır. Aynı paragrafın Türkçesini ve İngilizcesini al. GPT-4'ün tokenizer'ıyla (tiktoken, "cl100k_base") ikisini de token'la, kelime başına kaç token düştüğünü yaz. Sonra aynı Türkçe paragrafı kendi tokenizer'ınla token'la. Türkçenin eklemeli yapısı bu sayıları nasıl etkiliyor?

In [54]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # GPT-4

tr = "Dün akşam arkadaşlarımla birlikte İstanbul'a gittik. Şehrin tarihi sokaklarında yürüdük, küçük bir kafede çay içtik ve eski kitapçıları gezdik. Evimize döndüğümüzde çok yorulmuştuk ama günümüzden çok memnunduk."
en = "Yesterday evening I went to Istanbul with my friends. We walked through the historic streets of the city, drank tea in a small cafe and visited the old bookshops. When we returned home we were very tired, but we were very happy with our day."

for ad, metin in [("Türkçe", tr), ("İngilizce", en)]:
    kelime = len(metin.split())
    token = len(enc.encode(metin))
    print(f"{ad}: {kelime} kelime, {len(metin.encode('utf-8'))} byte, {token} token, kelime başına {token / kelime:.2f} token")

Türkçe: 27 kelime, 239 byte, 89 token, kelime başına 3.30 token
İngilizce: 45 kelime, 241 byte, 51 token, kelime başına 1.13 token


In [55]:
# GPT-4 bazı kelimeleri nasıl parçalıyor?
for kelime in ["arkadaşlarımla", "sokaklarında", "yorulmuştuk", "günümüzden", "friends", "streets", "bookshops"]:
    print(kelime, "->", [enc.decode([t]) for t in enc.encode(" " + kelime)])


normal_bpe = encode(tr)
print()
print("GPT-4 token sayısı (Türkçe):", len(enc.encode(tr)))
print("normal BPE token sayısı (Türkçe):", len(normal_bpe))
print(f"normal BPE kelime başına: {len(normal_bpe) / len(tr.split()):.2f} token")
print(f"normal BPE sıkıştırma: {len(tr.encode('utf-8')) / len(normal_bpe):.2f}X")

arkadaşlarımla -> [' ark', 'ada', 'ş', 'ları', 'ml', 'a']
sokaklarında -> [' sok', 'ak', 'lar', 'ında']
yorulmuştuk -> [' y', 'or', 'ul', 'mu', 'şt', 'uk']
günümüzden -> [' gün', 'üm', 'üz', 'den']
friends -> [' friends']
streets -> [' streets']
bookshops -> [' book', 'shops']

GPT-4 token sayısı (Türkçe): 89
normal BPE token sayısı (Türkçe): 184
normal BPE kelime başına: 6.81 token
normal BPE sıkıştırma: 1.30X


In [56]:
# GPT-4 deseni ile Türkçe paragrafı 89 token'a çevirdi, İngilizce paragrafı ise 51 token'a çevirdi. Kelime başına Türkçe 3.30, İngilizce'ye ise 1.13 token düşüyor.
# Türkçe paragraf İngilizceden yaklaşık 1.75 kat daha fazla token tutuyor.
#
# Kelime sayısı farklı çıktı (Türkçe 27, İngilizce 45) çünkü Türkçede ekler kelimeye yapışıyor. "arkadaşlarımla" tek kelime ama İngilizcede "with my friends" diye 3 kelime.
# Yani Türkçe kelime başına daha çok anlam taşıyor, ama yine de daha çok token'a bölünüyor.
#
# Bunun sebebi GPT-4'ün çoğunlukla İngilizce metinle eğitilmiş olması. "friends" gibi İngilizce kelimeler tek token oluyor.
# Türkçe kelimeler ise kök ve eklere bölünüyor, mesela "sokaklarında" 4 parça, "arkadaşlarımla" 6 parça oluyor.
# Ek kombinasyonları çok fazla olduğu için kelime parçalanıyor.
#
# Normal BPE ile aynı Türkçe paragraf 184 token oldu, GPT-4'ün ise 89 token.
# Çünkü normal BPE'de sadece 20 birleştirme var, GPT-4'te yaklaşık 100 bin token var.
#
# Sonuç olarak Türkçe eklemeli bir dil olduğu için aynı metin daha çok token tutuyor, bu da daha pahalı ve daha az metnin sığması demek oluyor.

## Görev 6

In [57]:
# Ek, erken bitirenler için, ikisinden biri. (a) GPT-4'ün birleştirmelerini yükle ve tokenizer'ının tiktoken'la birebir aynı token'ları verdiğini göster, minbpe alıştırmasının 3. adımı. (1:11:38 - 1:25:28) (b) Geçen haftaki GPT'yi kendi tokenizer'ınla Türkçe metinde eğit. Ürettiği metinden örnek ver. Val loss'u harf seviyesindeki modelinkiyle doğrudan karşılaştırabilir misin, neden? (1:43:27 - 1:48:11)

In [58]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
mergeable_ranks = enc._mergeable_ranks  
print("bilinen token sayısı:", len(mergeable_ranks))

def bpe(mergeable_ranks, token, max_rank):
    
    parts = [bytes([b]) for b in token]
    while True:
        min_idx = None
        min_rank = None
        for i, pair in enumerate(zip(parts[:-1], parts[1:])):
            rank = mergeable_ranks.get(pair[0] + pair[1])
            if rank is not None and (min_rank is None or rank < min_rank):
                min_idx = i
                min_rank = rank
        if min_rank is None or (max_rank is not None and min_rank >= max_rank):
            break
        parts = parts[:min_idx] + [parts[min_idx] + parts[min_idx + 1]] + parts[min_idx + 2:]
    return parts

def recover_merges(mergeable_ranks):
    
    merges = {}
    for token, rank in mergeable_ranks.items():
        if len(token) == 1:
            continue 
        pair = tuple(bpe(mergeable_ranks, token, max_rank=rank))
        assert len(pair) == 2
        merges[(mergeable_ranks[pair[0]], mergeable_ranks[pair[1]])] = rank
    return merges

gpt4_merges = recover_merges(mergeable_ranks)
print("birleştirme sayısı:", len(gpt4_merges))

bilinen token sayısı: 100256
birleştirme sayısı: 100000


In [59]:
byte_shuffle = {i: mergeable_ranks[bytes([i])] for i in range(256)}
print(list(byte_shuffle.items())[:5])

[(0, 188), (1, 189), (2, 190), (3, 191), (4, 192)]


In [60]:
def encode_chunk(chunk_bytes):
    ids = [byte_shuffle[b] for b in chunk_bytes]   # byte'ları GPT-4'ün sırasına çevir
    while len(ids) >= 2:
        stats = get_stats(ids)
        pair = min(stats, key=lambda p: gpt4_merges.get(p, float("inf")))   # en erken öğrenilen birleştirme
        if pair not in gpt4_merges:
            break
        ids = merge(ids, pair, gpt4_merges[pair])
    return ids

def gpt4_encode(text):
    ids = []
    for chunk in re.findall(gpt4pat, text):   # önce GPT-4 deseniyle parçalara böl
        ids.extend(encode_chunk(chunk.encode("utf-8")))
    return ids

In [61]:
tests = {"Türkçe": tr, "İngilizce": en, "emoji": "Bugün hava çok güzel! 😊🌞",
         "kod": "for i in range(10):\n    print(i)\n", "kesme": "Ankara'dan İstanbul'da Bedia'yı"}
for ad, metin in tests.items():
    a, b = gpt4_encode(metin), enc.encode(metin)
    print(f"{ad:10} benim: {len(a):3d} token, tiktoken: {len(b):3d} token, aynı mı: {a == b}")

parca = text_tr[:30000]   
a, b = gpt4_encode(parca), enc.encode(parca)
print(f"corpus: {len(a)} token, aynı mı: {a == b}")

Türkçe     benim:  89 token, tiktoken:  89 token, aynı mı: True
İngilizce  benim:  51 token, tiktoken:  51 token, aynı mı: True
emoji      benim:  13 token, tiktoken:  13 token, aynı mı: True
kod        benim:  11 token, tiktoken:  11 token, aynı mı: True
kesme      benim:  13 token, tiktoken:  13 token, aynı mı: True
corpus: 12277 token, aynı mı: True
